# HuggingFace Transformers Text Classification with Progression Tracking

This example demonstrates how to fine-tune a transformer model for text classification using the [IMDB movie review dataset](https://huggingface.co/datasets/imdb) and [HuggingFace Transformers](https://huggingface.co/docs/transformers).

This notebook walks you through:
- Running training on Kubernetes with automatic progression tracking
- Mounting persistent storage (RWX PVC) for efficient model/dataset caching
- Using rank 0 to download once, then all workers load from shared storage
- Scaling across multiple nodes with distributed training
- Monitoring training progress in real-time without any extra code

**Key Feature**: TransformersTrainer automatically tracks training progress (steps, epochs, metrics) without requiring any modifications to your training code!


## Install the Kubeflow SDK

Install the Kubeflow SDK interact with Kubeflow Trainer APIs:

In [3]:
%pip install --force-reinstall ../dist/kubeflow-0.2.0-py3-none-any.whl

Processing /Users/abdhumal/Dev/RedHatDev/sdk/dist/kubeflow-0.2.0-py3-none-any.whl
  Using cached kubeflow_katib_api-0.19.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached kubeflow_trainer_api-2.1.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached kubernetes-34.1.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pydantic-2.12.4-py3-none-any.whl.metadata (89 kB)
  Using cached certifi-2025.11.12-py3-none-any.whl.metadata (2.5 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached google_auth-2.43.0-py2.py3-none-any.whl.metadata (6.6 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached urllib3-2.3.0-py3-none-any.w

## Define the Training Function

Create a function to fine-tune DistilBERT on the IMDB sentiment classification task.

**Note**: This is standard HuggingFace Transformers code with NO special instrumentation needed for progression tracking!


In [4]:
def train_sentiment_classifier():
    """Only rank 0 downloads model/dataset to shared storage, other ranks wait."""
    import os
    import torch.distributed as dist
    from datasets import load_dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

    # Get rank (0 for single-node or first worker in multi-node)
    rank = int(os.environ.get("RANK", 0))
    
    model_path = "/workspace/model"
    dataset_path = "/workspace/dataset"
    
    # Only rank 0 downloads to shared storage
    if rank == 0:
        print(f"[Rank {rank}] Downloading model and dataset to shared storage...")
        
        # Download model if not exists
        if not os.path.exists(model_path):
            model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
            tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
            model.save_pretrained(model_path)
            tokenizer.save_pretrained(model_path)
            print(f"[Rank {rank}] Model saved to {model_path}")
        
        # Download dataset if not exists
        if not os.path.exists(dataset_path):
            dataset = load_dataset("stanfordnlp/imdb")
            dataset.save_to_disk(dataset_path)
            print(f"[Rank {rank}] Dataset saved to {dataset_path}")
    
    # Synchronize: all ranks wait for rank 0 to finish downloading
    if dist.is_initialized():
        dist.barrier()
        print(f"[Rank {rank}] Barrier passed, loading from shared storage...")
    
    # All ranks load from shared storage
    from datasets import load_from_disk
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    dataset = load_from_disk(dataset_path)
    
    # Tokenize and prepare data
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    
    train_dataset = dataset["train"].select(range(1000)).map(tokenize_function, batched=True)
    eval_dataset = dataset["test"].select(range(200)).map(tokenize_function, batched=True)

    # Train
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="./results",
            num_train_epochs=3,
            learning_rate=2e-5,
            logging_steps=10,
            per_device_train_batch_size=8,
        ),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )
    trainer.train()
    print(f"Results: {trainer.evaluate()}")


## Run Training with PVC (Rank 0 Downloads)

This approach mounts a **ReadWriteMany (RWX) PVC** to `/workspace`:

**How it works**:
1. **PVC mounted at `/workspace`** - shared storage across all worker pods
2. **Rank 0 downloads** model and dataset to `/workspace/model` and `/workspace/dataset`
3. **Other ranks wait** (via `dist.barrier()`) then load from shared storage
4. **Progression tracking** automatically enabled (tracks steps, epochs, metrics)

**To use**: Replace `"rwx-pvc-name"` with your actual PVC name. On subsequent runs, rank 0 will skip downloading if files exist.


In [ ]:
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.options import ContainerOverride, PodSpecOverride, PodTemplateOverride, PodTemplateOverrides

# Initialize Kubernetes backend (default)
from kubeflow.trainer.backends.kubernetes.backend import KubernetesBackendConfig
from kubernetes import client

api_server = ""
token = ""

configuration = client.Configuration()
configuration.host = api_server
configuration.api_key = {"authorization": f"Bearer {token}"}

# Un-comment if your cluster API server uses a self-signed certificate or an un-trusted CA
configuration.verify_ssl = False

api_client = client.ApiClient(configuration)
trainer_client = TrainerClient(backend_config= KubernetesBackendConfig(client_configuration=api_client.configuration))

print("Available runtimes :", len(trainer_client.list_runtimes()))
for r in trainer_client.list_runtimes():
    print(f"- {r.name}")


/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Available runtimes : 6


/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


- torch-cuda-241
- torch-cuda-251
- torch-cuda-251-with-pvc
- torch-rocm-241
- torch-rocm-251
- traininghub-wrapper-runtime


/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advi

In [6]:
# Submit training job with TransformersTrainer
# Progression tracking is ENABLED BY DEFAULT (no extra code needed!)
job_name = trainer_client.train(
    trainer=TransformersTrainer(
        func=train_sentiment_classifier,
        packages_to_install=["torch", "transformers", "datasets", "accelerate"],
        # enable_progression_tracking=True,  # This is the default!
        # metrics_port=28080,  # Default port for metrics server
        # metrics_poll_interval_seconds=30,  # Default: 30 seconds, How often to poll metrics
    ),
    runtime=trainer_client.get_runtime("torch-cuda-251-with-pvc"),
    # Mount RWX PVC at /workspace - rank 0 downloads model/dataset, other ranks load from it
    options=[
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    volumes=[{"name": "workspace", "persistentVolumeClaim": {"claimName": "rwx-pvc-name"}}],
                    containers=[ContainerOverride(name="node", volume_mounts=[{"name": "workspace", "mountPath": "/workspace"}])]
                )
            )
        )
    ]
)

/Users/abdhumal/.pyenv/versions/3.12.11/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.abdhumal-osd.qic7.p1.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [7]:
# check job status
job = trainer_client.get_job(job_name)
print(f"Final TrainJob Status:")
print(f"   Name: {job.name}")
print(f"   Status: {job.status}")
print(f"   Created: {job.creation_timestamp}")
print(f"   Nodes: {job.num_nodes}")
print(f"   Runtime: {job.runtime.name}")

if job.steps:
    print(f"   Steps:")
    for step in job.steps:
        print(f"     - {step.name}: {step.status}")
    print()

Final TrainJob Status:
   Name: we21c6c63913
   Status: Running
   Created: 2025-11-19 08:52:21+00:00
   Nodes: 1
   Runtime: torch-cuda-251-with-pvc
   Steps:
     - node-0: Running



## Progression Tracking

TransformersTrainer automatically tracks training progress (steps, epochs, loss, metrics) via HTTP endpoint on port 28080. No code changes needed!

### Fetch Metrics from Rank 0 Pod

Query the metrics endpoint directly from the rank 0 training pod:


In [13]:
from kubernetes import client as k8s_client, config
import json

config.load_kube_config()
custom_api = k8s_client.CustomObjectsApi()

trainjob_name = "we21c6c63913"
job_namespace = "default"

try:
    trainjob = custom_api.get_namespaced_custom_object(
        group="trainer.kubeflow.org",
        version="v1alpha1",
        namespace=job_namespace,
        plural="trainjobs",
        name=trainjob_name
    )
    
    annotations = trainjob.get('metadata', {}).get('annotations', {})
    
    print(f"TrainJob Annotations for {trainjob_name}:\n")
    
    # Progression tracking config
    print("Config:")
    print(f"  tracking-enabled: {annotations.get('trainer.opendatahub.io/progression-tracking', 'N/A')}")
    print(f"  metrics-port: {annotations.get('trainer.opendatahub.io/metrics-port', 'N/A')}")
    print(f"  poll-interval: {annotations.get('trainer.opendatahub.io/metrics-poll-interval', 'N/A')}s")
    
    # Progression metrics (if controller populated them)
    if 'trainer.opendatahub.io/progress-percentage' in annotations:
        print("\nMetrics:")
        print(f"  progress: {annotations.get('trainer.opendatahub.io/progress-percentage', 'N/A')}%")
        print(f"  step: {annotations.get('trainer.opendatahub.io/current-step', 'N/A')}/{annotations.get('trainer.opendatahub.io/total-steps', 'N/A')}")
        print(f"  epoch: {annotations.get('trainer.opendatahub.io/current-epoch', 'N/A')}/{annotations.get('trainer.opendatahub.io/total-epochs', 'N/A')}")
        print(f"  remaining: {annotations.get('trainer.opendatahub.io/estimated-remaining-seconds', 'N/A')}s")
    print(f"\nAll annotations:\n{json.dumps(annotations, indent=2)}")
    
except Exception as e:
    print(f"Error: {e}")

TrainJob Annotations for we21c6c63913:

Config:
  tracking-enabled: true
  metrics-port: 28080
  poll-interval: 30s

All annotations:
{
  "reconcile": "true",
  "trainer.opendatahub.io/metrics-poll-interval": "30",
  "trainer.opendatahub.io/metrics-port": "28080",
  "trainer.opendatahub.io/progression-tracking": "true",
  "trainer.opendatahub.io/trainerStatus": "{\"progressPercentage\":100,\"estimatedRemainingSeconds\":0,\"estimatedRemainingTimeSummary\":\"complete\",\"currentStep\":366,\"totalSteps\":375,\"currentEpoch\":2,\"totalEpochs\":3,\"trainMetrics\":{\"grad_norm\":0.011240859515964985,\"learning_rate\":8.533333333333334e-7,\"loss\":0.0005},\"lastUpdatedTime\":\"2025-11-19T09:01:38Z\"}"
}


In [ ]:
trainer_client.delete_job(job_name)